# **Lab 5: Regression**

Given two numerical quantities that move together, what is the best straight
line through them, and how do you tell whether a line was the right shape at
all?

You will build five short functions, then use them on a completely different
dataset to see where the line works and where it does not.

There are no hidden tests. Every check you can see is every check there is.

#### **Helpful Resource:**
- [Python Reference](https://ulwazi.wits.ac.za/courses/89081/pages/detailed-python-reference-sheet-python-cheat-sheet-2?module_item_id=1200407)

**Recommended Readings:**
- [15.2 The Regression Line](https://inferentialthinking.com/chapters/15/2/Regression_Line.html)
and
- [15.6 Numerical Diagnostics](https://inferentialthinking.com/chapters/15/6/Numerical_Diagnostics.html).


## **Do not change the code in the two cells below** ##

In [ ]:
# Run this cell first to install the libraries. It may take a minute.
%pip install -q urllib3<2.0 otter-grader==7.0.0 datascience ipywidgets

In [ ]:
# Run this cell to set up the notebook, but please don't change it.

try:
    import pyodide_http
    pyodide_http.patch_all()
except ImportError:
    pass

import numpy as np
from datascience import *

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plots
plt = plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter('ignore', FutureWarning)

import hashlib
import otter
grader = otter.Notebook("lab5.ipynb")

---
---

### **If something goes wrong** (Refer to this if get an error)

Errors are normal. Everyone gets them, including people who have been doing this
for years. The message tells you what happened, once you know how to read it.

| What you see | What it usually means |
|---|---|
| `NameError: name 'x' is not defined` | You have not run the cell that creates `x`, or you typed it differently |
| `NameError: name 'Table' is not defined` | The setup cell at the very top has not been run. That is the cell that brings in `Table`, `np` and the plotting tools. Run it, then run your own cell again |
| `NameError: name 'grader' is not defined` | The same cause, for the cell that sets up the tests. If you did run it, read its output: the `otter` install needs a connection the first time, and if it failed then every `grader.check` below it fails too. Run that cell again |
| `ValueError: The column "X" is not in the table` | You typed a column name that does not exist. The message lists the real ones, so read to the end of it |
| `TypeError: unsupported operand ... 'NoneType'` | A function you wrote did not `return` anything, so it handed back nothing. Check that it has a `return` line |
| `Got nothing` in a test result | The same cause: a missing `return` |
| `TypeError: unsupported operand` (any other) | Arithmetic on text, or on a whole table where you wanted one column |
| `SyntaxError: invalid character` | You pasted a character Python does not accept. Dashes and quote marks copied out of formatted text often look right but are not. Retype the line by hand |
| `SyntaxError` (any other) | A missing bracket or quote, often on the line **above** the one it points at |
| `IndexError` | You asked for item 5 of something with fewer than 6 things in it |

**If several cells stop making sense at once**, the cause is almost always that
they were run out of order. Choose **Kernel** then **Restart Kernel and Run All Cells**, and
work down from the top again.

Try things. Running something wrong and reading the error is how this is
learned, and you cannot break anything.

---

## **1. The toolkit**

We will predict a footballer's market value from their age. The data is 100
players from FIFA 22.

In [ ]:
# Run this cell to load the data
fifa = Table.read_table('fifa22.csv')

# Select a subset of columns to analyze (there are 110 columns in the original dataset)
fifa = fifa.select("short_name", "overall", "value_eur", "wage_eur", "age", "pace", "shooting", "passing", "attacking_finishing")
fifa.show(5)

---

### **First, once, by hand**

Before writing anything reusable, here is the calculation done for one pair of
columns with nothing hidden. Run both cells and read them. Every question in
this section asks you to turn part of this into a function.

In [ ]:
# Just run this cell. Converting one array to standard units.
ages = fifa.column('age')
age_su = (ages - np.mean(ages)) / np.std(ages)

print('mean of age in standard units:', round(np.mean(age_su), 10))
print('SD of age in standard units:  ', round(np.std(age_su), 10))

In standard units the mean is 0 and the SD is 1, whatever you started with. That
is the point of the conversion: it removes the units, so two quantities on
completely different scales become comparable.

One detail that matters later. `np.std` divides by the number of values, which
gives what is called the **population** standard deviation. Many calculators,
and some other software, divide by one less than the number of values instead
and give a slightly larger answer. Use `np.std` exactly as written above, with
no extra arguments, or your answers will not match the ones the tests expect.

In [ ]:
# Just run this cell. Correlation between age and value, written out in full.
values = fifa.column('value_eur')
value_su = (values - np.mean(values)) / np.std(values)

r = np.mean(age_su * value_su)
r

Negative, and strongly so.

Be careful about what that shows. This file is the first 100 rows of the game's
players sorted from the highest overall rating downwards, so everyone in it is
rated 84 or above. That is a cut at row 100 rather than a rating threshold: the
full game contains other players rated 84 who did not make the cut. Among an
already exceptional group like this one, value tracks how many years a player
has left. On a dataset covering all players the picture would look different. A
relationship found in a narrow slice of the data does not automatically hold
outside it.

The line through these points has a slope and an intercept, and both come
straight from the correlation. Run the next cell to see them worked out.

In [ ]:
# Just run this cell. The regression line for age and value, written out in full.
# Here age is x, the quantity we predict from, and value is y, the quantity we
# predict.
slope_by_hand = r * np.std(values) / np.std(ages)
intercept_by_hand = np.mean(values) - slope_by_hand * np.mean(ages)

print('slope:    ', round(slope_by_hand, 2))
print('intercept:', round(intercept_by_hand, 2))

Those two numbers are all you need to make a prediction. For any age `x`, the
predicted value is

$$\text{prediction} = \text{slope} \times x + \text{intercept}$$

which is the equation of a straight line. Run the cell below to use it.

In [ ]:
# Just run this cell. Using the slope and intercept to predict.
predicted_values = slope_by_hand * ages + intercept_by_hand

print('first player is aged', ages.item(0))
print('   we predict a value of', round(predicted_values.item(0)))
print('   their actual value is', values.item(0))

# The same arithmetic works for one number as for a whole array.
print()
print('predicted value for a 30-year-old:', round(slope_by_hand * 30 + intercept_by_hand))

Those two lines are the whole of the regression line:

$$\text{slope} = r \times \frac{\text{SD of } y}{\text{SD of } x}
\qquad
\text{intercept} = \text{mean of } y - \text{slope} \times \text{mean of } x$$

The slope is the correlation, rescaled from standard units back into the original
units. The intercept then shifts the line so it passes through the **point of
averages**, the point whose x is the mean of x and whose y is the mean of y. Chapter 15.2 of the textbook derives both if you want the reasoning.

---

### **Writing a function**

Everything above is written for two particular columns. A **function** lets you
write a calculation once and then use it on any columns you like. The next few
cells are the whole of the syntax you need. Run each one and read it.

**`return` is what makes a function produce an answer.** A function without it
runs and then hands back nothing. That is the `NoneType` row in the error table
at the top of this lab.

In [ ]:
# Just run this cell.
def double(number):
    return number * 2      # 'return' hands the answer back to whoever called

print('double(7) is', double(7))

**A function can call another function you wrote earlier.** From Question 1.2
onwards you will call the functions you wrote in the earlier questions, in
exactly this way.

In [ ]:
# Just run this cell.
def quadruple(number):
    return double(double(number))

print('quadruple(7) is', quadruple(7))

**The same function works on a whole array and on a plain list.** It does not
have to do anything special for that: `np.mean` gives back a single number, and
numpy turns the list into an array to do the subtraction. Some of the tests
below hand your functions a plain list, so this matters.

In [ ]:
# Just run this cell.
def centre(data):
    return data - np.mean(data)

print('centre of an array:', centre(make_array(1, 2, 3)))
print('centre of a list: ', centre([1, 2, 3]))

**A function can take a column name as an argument.** The name arrives as a
string inside the variable `col`, so you write `tbl.column(col)`, with no quote
marks, because `col` is a variable, not the name itself. Question 1.5 works like
this.

In [ ]:
# Just run this cell.
def column_mean(tbl, col):
    return np.mean(tbl.column(col))

print('mean age:', round(column_mean(fifa, 'age'), 2))

Now write the calculation from the cells above as functions, so it works for any
two columns rather than only these. **The arithmetic is already above; what you
are adding is the ability to reuse it.**

---

**Question 1.1.** Define a function `standard_units` that converts a given array to standard units.

*Hint:* You may find the `np.mean` and `np.std` functions helpful. The
arithmetic is in the first "just run this cell" above.

In [ ]:
def standard_units(data):
    return ...

In [ ]:
grader.check("q1_1")

---

**Question 1.2.** Define a function `correlation` that computes the correlation between `x` and `y`, which are 2 arrays of data in their original units.

*Hint:* Call your own `standard_units` from inside this function, the way
`quadruple` called `double` above.

In [ ]:
def correlation(x, y):
    return ...

In [ ]:
grader.check("q1_2")

---

**Question 1.3.** Define a function `slope` that computes the slope of our line of best fit (to predict y given x). The function takes in x and y, which are two arrays of data in their original units. Assume we want to create a line of best fit in original units.

*Hint:* `r` below is the correlation between `x` and `y`. You have just written a
function that computes it.

In [ ]:
def slope(x, y):
    r = ...
    return ...

In [ ]:
grader.check("q1_3")

---

**Question 1.4.** Define a function `intercept` that computes the intercept of our line of best fit (to predict y given x), given 2 arrays of data in original units. Assume we want to create a line of best fit in original units.

*Hint:* `b` below is the slope of the same line. You have just written a function
that computes it, so call it here rather than working the slope out again.

In [ ]:
def intercept(x, y):
    b = ...
    return ...

In [ ]:
grader.check("q1_4")

---

**Question 1.5.** Define a function `predict` that takes in a table and 2 column names as strings, and returns an array of predictions. The predictions should be created using a fitted **regression line**. We are predicting `col2` from `col1`, both in original units.

*Hint 1:* `col1` and `col2` arrive as strings, so write `tbl.column(col1)`, as in
the `column_mean` example above.

*Hint 2:* Once you have the two arrays, the last line is the prediction formula
from the cells above, with your `slope` and `intercept` functions in place of the
numbers.

*Hint 3:* Re-reading [15.2](https://inferentialthinking.com/chapters/15/2/Regression_Line.html#the-regression-line) might be helpful here.

In [ ]:
def predict(tbl, col1, col2):
    x = ...
    y = ...
    return ...

In [ ]:
grader.check("q1_5")

---
---

## **2. A second dataset, and a harder question**

Your toolkit is not specific to footballers. Here it is on Old Faithful, a hot
spring in the United States that shoots water into the air at fairly regular
intervals. Each row is one eruption: how long it lasted, and how
long until the next one.

A line that fits is only part of the answer. This section asks whether a line was
the **right shape**, and where you can trust what it predicts. You will compute
the predictions, then the errors the line makes, then read those errors and say
what they show.

You will need one new table method here, and one piece of array arithmetic. Run
the cell below to see both.

In [ ]:
# Just run this cell. Adding a column to a table, and subtracting two arrays.
demo = Table().with_columns('x', make_array(1, 2, 3))
demo_with_squares = demo.with_columns('x squared', demo.column('x') ** 2)

# Subtracting one array from another works entry by entry and gives back an
# array of the same length. No loop is needed.
gap = demo_with_squares.column('x squared') - demo_with_squares.column('x')
print('x squared minus x:', gap)

demo_with_squares

`tbl.with_columns(name, values)` returns a **new** table with that column added.
It does not change the original, so you have to assign the result to something.
The name is a string and the values are an array with one entry per row.

The subtraction works the same way for any two arrays of equal length, including
two columns of the same table. You will use this in Question 2.2.

In [ ]:
# Just run this cell
# The file's columns are called 'eruptions' and 'waiting'; renamed here to the
# shorter names the questions below use.
faithful = (Table.read_table('faithful.csv')
            .relabeled('eruptions', 'duration')
            .relabeled('waiting', 'wait'))
faithful.scatter('duration', 'wait', fit_line=True)
plots.show()
faithful.show(3)

---

**Question 2.1.** Make predictions for the waiting time after each eruption in the `faithful` table.  (Of course, we know exactly what the waiting times were!  We are doing this so we can see how accurate our predictions are.)

Use your `predict` function from Question 1.5. Put the results into a new table
called `faithful_predictions` that keeps both of the columns `faithful` already
has and adds the predictions as a third column called `predicted wait`, in this
order:

|duration|wait|predicted wait|
|-|-|-|
|3.6|79|72.1011|

The table above is rounded to four decimal places so that it fits on the page.
**Do not round your own answer.** Question 2.2 builds on this one and needs the
full precision.

*Hint:* Your answer can be just one line, using `with_columns` as in the cell
above.  There is no need for a `for` loop; use array arithmetic instead.

In [ ]:
faithful_predictions = ...
faithful_predictions

In [ ]:
grader.check("q2_1")

---

**Question 2.2.** How close were we?  Compute the *residual* for each eruption in the dataset.  The residual is the actual waiting time minus the predicted waiting time, so a positive residual means the line predicted too low.  Add the residuals to `faithful_predictions` as a new column called `residual` and name the resulting table `faithful_residuals`.

As mentioned in [Chapter 15.6](https://inferentialthinking.com/chapters/15/6/Numerical_Diagnostics.html?highlight=residuals#average-of-residuals), a useful property of the residuals produced by linear regression with an intercept is that they always sum to zero.  What you actually get will be a very small number rather than exactly zero, because of the way computers store decimals.  You can check your work by running this line in a scratch cell:

```python
sum(faithful_residuals.column('residual'))
```

*Hint:* residual = actual value - predicted value. Subtracting one array from
another, as in the demonstration above, does this for every row at once, so your
code will be much simpler if you don't use a `for` loop.

In [ ]:
residuals = ...
faithful_residuals = ...
faithful_residuals

In [ ]:
grader.check("q2_2")

Here is a plot of the residuals you computed.  Each point corresponds to one eruption.  It shows how much our prediction over- or under-estimated the waiting time.

In [ ]:
faithful_residuals.scatter("duration", "residual", color="r")

The plot shows two separate groups of points, or **clouds**, because eruption
durations cluster into short ones and long ones. The cell below splits the eruptions at 3 minutes and
compares, within each group, the spread of the waiting times against the spread
of what the line got wrong. Run it.

In [ ]:
# Just run this cell. How much does the line actually explain?
short = faithful_residuals.where('duration', are.below(3))
long = faithful_residuals.where('duration', are.above_or_equal_to(3))

print('group            n     SD of wait   SD of residual')
for name, group in [('all eruptions', faithful_residuals), ('short (< 3 min)', short), ('long (>= 3 min)', long)]:
    print('{:16} {:4}   {:9.2f}   {:12.2f}'.format(
        name, group.num_rows,
        np.std(group.column('wait')),
        np.std(group.column('residual'))))

print()
print('correlation, all eruptions: ', round(correlation(faithful_residuals.column('duration'),
                                                        faithful_residuals.column('wait')), 2))
print('correlation, short only:    ', round(correlation(short.column('duration'),
                                                        short.column('wait')), 2))
print('correlation, long only:     ', round(correlation(long.column('duration'),
                                                        long.column('wait')), 2))

So a straight line has no strong overall bend here, and it does remove a lot of
the spread: across all the eruptions, the SD of the waiting time is about 13.6
minutes and the SD of the residuals is about 5.9.

Inside either cluster it removes almost nothing. The residual SD in each group is
about the same as the wait SD in that group, and the correlation within a group
is around 0.3, against 0.90 overall. Nearly all of that strong overall
correlation comes from the gap between the two clusters, not from the
relationship inside either one.

Both clouds also tilt the same way, downwards. That is one systematic effect
rather than two small quirks, and it is the same fact seen from another angle:
the fitted slope of about 10.7 minutes of waiting per minute of eruption is
roughly twice the slope you would get inside either group on its own.

A single line is therefore a fair summary of the two-cluster pattern, and a poor
description of what happens within a cluster.

In [ ]:
# Just run this cell -- the line, and a helper for printing predictions
faithful_slope = slope(faithful.column('duration'), faithful.column('wait'))
faithful_intercept = intercept(faithful.column('duration'), faithful.column('wait'))

print('slope:', round(faithful_slope, 4))
print('intercept:', round(faithful_intercept, 4))

def print_prediction(duration, predicted_waiting_time):
    print("After an eruption lasting", duration,
          "minutes, we predict you'll wait", predicted_waiting_time,
          "minutes until the next eruption.")

---

**Question 2.3.** In `faithful`, no eruption lasted exactly 0, 2.5, or 60 minutes.  Using the line whose slope and intercept were just printed, what is the predicted waiting time for an eruption that lasts 0 minutes?  2.5 minutes?  An hour?

*Hint:* Use `faithful_slope` and `faithful_intercept` from the cell above.

In [ ]:
zero_minute_predicted_waiting_time = ...
two_point_five_minute_predicted_waiting_time = ...
hour_predicted_waiting_time = ...

print_prediction(0, zero_minute_predicted_waiting_time)
print_prediction(2.5, two_point_five_minute_predicted_waiting_time)
print_prediction(60, hour_predicted_waiting_time)

In [ ]:
grader.check("q2_3")

Those are three different situations, not three versions of the same one.

**0 minutes** is outside the data. The shortest eruption in the table lasts 1.6
minutes. The line was fitted where the data is, and 0 is not there.

**60 minutes** is the same problem taken much further, and the answer is
nonsense: about 677 minutes, more than eleven hours. The line will still return
a number however far outside the data you go, and it will not warn you.

**2.5 minutes** is the interesting one, because it is *not* outside the range at
all. Durations run from 1.6 to 5.1 minutes, so 2.5 sits comfortably inside them.
But only 5 of the 272 eruptions lasted between 2.5 and 3 minutes: that stretch is
the gap between the two clusters. The prediction there is 60.3 minutes, while the
eruptions that did last between 2.3 and 2.7 minutes had waits ranging from 47 to
71. The prediction is not absurd, it is unsupported, which is a different
problem. Being inside the range of the data is not the same as being where the
data is.

---

**Question 2.4.** Look again at the residual plot and at the numbers printed
below it. Which one of these statements is best supported by them? Set `q2_4` to
`1`, `2`, `3` or `4`.

1. The line is a poor fit, because some of the residuals are more than 10 minutes.
2. Within each of the two clusters the line explains almost none of the variation in waiting time, and the strong overall correlation comes mostly from the gap between the clusters.
3. The residual plot bends sharply, which shows the relationship is curved and a straight line should not be used at all.
4. The residuals sum to zero, which shows the line is the correct model for this data.

In [ ]:
q2_4 = ...

In [ ]:
grader.check("q2_4")

---

## **You're done!**

**Important submission information:**
- **Run all the tests** and verify that they all pass
- **Save** from the **File** menu
- **Run the final cell to generate the grader.check_all()**
- **Click the download button to download the .ipynb file (see the figure below)**
- **Then, go to Ulwazi and submit the .ipynb file to Lab 5: Regerssion**

**It is your responsibility to make sure your work is saved before running the last cell.**


<img src="download_ipynb.png" alt="download ipynb" width="800"/>

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()

## **Submission**

Make sure you have run all cells in your notebook in order before clicking the download button to download the .ipynb file. **Please save your work before downloading!**